In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv('/Users/nitesh/Downloads/Task 1/olist_orders_dataset.csv')

print(orders.head())
print(orders.shape)
print(orders.info())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [ ]:
# Checking data Quality

In [2]:
print(orders.isnull().sum())
print("Duplicate rows:", orders.duplicated().sum())
print(orders.dtypes)

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Duplicate rows: 0
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


An initial quality assessment was performed using Pandas to identify missing values, duplicate records, and data types. No duplicate rows were found in the dataset. Missing values were identified in the order approval, carrier delivery, and customer delivery timestamp fields. The customer delivery timestamp contained the highest number of missing values, with 2,965 records. All timestamp fields were initially stored as object/string data types and therefore required conversion to datetime format before performing delivery-time calculations.

In [ ]:
# Convert the dates 

In [3]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print(orders[date_cols].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


All five timestamp columns are now correctly converted from object to datetime64[ns].

In [ ]:
# Understanding the missing dates 

In [4]:
print(
    orders.groupby("order_status")[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ].apply(lambda x: x.isna().sum())
)

              order_approved_at  order_delivered_carrier_date  \
order_status                                                    
approved                      0                             2   
canceled                    141                           550   
created                       5                             5   
delivered                    14                             2   
invoiced                      0                           314   
processing                    0                           301   
shipped                       0                             0   
unavailable                   0                           609   

              order_delivered_customer_date  
order_status                                 
approved                                  2  
canceled                                619  
created                                   5  
delivered                                 8  
invoiced                                314  
processing                 

In [5]:
print("Delivered orders:", (orders["order_status"] == "delivered").sum())

print(
    "Delivered orders with missing customer delivery date:",
    orders.loc[
        orders["order_status"] == "delivered",
        "order_delivered_customer_date"
    ].isna().sum()
)

print(
    "Delivered orders with missing purchase date:",
    orders.loc[
        orders["order_status"] == "delivered",
        "order_purchase_timestamp"
    ].isna().sum()
)

print(
    "Delivered orders with missing estimated date:",
    orders.loc[
        orders["order_status"] == "delivered",
        "order_estimated_delivery_date"
    ].isna().sum()
)

Delivered orders: 96478
Delivered orders with missing customer delivery date: 8
Delivered orders with missing purchase date: 0
Delivered orders with missing estimated date: 0


In [ ]:
# Create the delivery-analysis dataset

In [6]:
delivery_orders = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
].copy()

print("Delivery analysis records:", len(delivery_orders))
print(delivery_orders.isna().sum())

Delivery analysis records: 96470
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      1
order_delivered_customer_date     0
order_estimated_delivery_date     0
dtype: int64


In [ ]:
# Actual Delivery dates 

In [7]:
delivery_orders["actual_delivery_days"] = (
    delivery_orders["order_delivered_customer_date"]
    - delivery_orders["order_purchase_timestamp"]
).dt.days

print(delivery_orders["actual_delivery_days"].describe())

count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: actual_delivery_days, dtype: float64


In [ ]:
# Estimated delivary days 

In [8]:
delivery_orders["estimated_delivery_days"] = (
    delivery_orders["order_estimated_delivery_date"]
    - delivery_orders["order_purchase_timestamp"]
).dt.days

print(delivery_orders["estimated_delivery_days"].describe())

count    96470.000000
mean        23.372748
std          8.758421
min          2.000000
25%         18.000000
50%         23.000000
75%         28.000000
max        155.000000
Name: estimated_delivery_days, dtype: float64


In [ ]:
# Delay days 

In [9]:
delivery_orders["delay_days"] = (
    delivery_orders["actual_delivery_days"]
    - delivery_orders["estimated_delivery_days"]
)

print(delivery_orders["delay_days"].describe())

count    96470.000000
mean       -11.279144
std         10.192137
min       -146.000000
25%        -16.000000
50%        -12.000000
75%         -7.000000
max        189.000000
Name: delay_days, dtype: float64


In [ ]:
# Delivary Performance 

In [10]:
delivery_orders["delivery_performance"] = delivery_orders[
    "delay_days"
].apply(
    lambda x: "Delayed" if x > 0 else "On Time"
)

print(delivery_orders["delivery_performance"].value_counts())

delivery_performance
On Time    89163
Delayed     7307
Name: count, dtype: int64


##### The average difference between actual and estimated delivery duration was -11.28 days, indicating that deliveries were generally completed earlier than the estimated date.

## Detect outliers using IQR

In [11]:
q1 = delivery_orders["actual_delivery_days"].quantile(0.25)
q3 = delivery_orders["actual_delivery_days"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Q1: 6.0
Q3: 15.0
IQR: 9.0
Lower bound: -7.5
Upper bound: 28.5


In [12]:
outliers = delivery_orders[
    (delivery_orders["actual_delivery_days"] < lower_bound) |
    (delivery_orders["actual_delivery_days"] > upper_bound)
]

print("Number of outliers:", len(outliers))
print("Percentage of outliers:", len(outliers) / len(delivery_orders) * 100)

Number of outliers: 5022
Percentage of outliers: 5.205763449777133


In [13]:
print(outliers["actual_delivery_days"].describe())

count    5022.000000
mean       40.047591
std        16.104355
min        29.000000
25%        31.000000
50%        36.000000
75%        43.000000
max       209.000000
Name: actual_delivery_days, dtype: float64


In [14]:
print(
    "Negative actual delivery days:",
    (delivery_orders["actual_delivery_days"] < 0).sum()
)

print(
    "Zero-day deliveries:",
    (delivery_orders["actual_delivery_days"] == 0).sum()
)

Negative actual delivery days: 0
Zero-day deliveries: 13


In [ ]:
# Creating flag for outliers 

In [15]:
delivery_orders["delivery_time_outlier"] = delivery_orders[
    "actual_delivery_days"
].apply(
    lambda x: "Outlier" if x > upper_bound else "Normal"
)

print(
    delivery_orders["delivery_time_outlier"].value_counts()
)

delivery_time_outlier
Normal     91448
Outlier     5022
Name: count, dtype: int64


Outlier Treatment: The Interquartile Range (IQR) method was used to identify unusually long delivery durations. The first quartile was 6 days and the third quartile was 15 days, resulting in an IQR of 9 days and an upper outlier threshold of 28.5 days. A total of 5,022 records (5.21%) were identified as outliers. These records were not removed because unusually long deliveries may represent genuine logistics delays or operational issues. Instead, an outlier flag was created so that these observations can be investigated separately during subsequent analysis.

In [ ]:
# Min-Max normalization

In [16]:
min_delivery = delivery_orders["actual_delivery_days"].min()
max_delivery = delivery_orders["actual_delivery_days"].max()

delivery_orders["actual_delivery_days_normalized"] = (
    (delivery_orders["actual_delivery_days"] - min_delivery)
    / (max_delivery - min_delivery)
)

print(
    delivery_orders[
        [
            "actual_delivery_days",
            "actual_delivery_days_normalized"
        ]
    ].head()
)

   actual_delivery_days  actual_delivery_days_normalized
0                     8                         0.038278
1                    13                         0.062201
2                     9                         0.043062
3                    13                         0.062201
4                     2                         0.009569


In [17]:
print(
    "Normalized minimum:",
    delivery_orders["actual_delivery_days_normalized"].min()
)

print(
    "Normalized maximum:",
    delivery_orders["actual_delivery_days_normalized"].max()
)

Normalized minimum: 0.0
Normalized maximum: 1.0


Normalization: Min-Max normalization was applied to the actual delivery duration feature. The transformation scales delivery duration to a range between 0 and 1 using the minimum and maximum observed delivery times. This creates a standardized numerical representation that can be useful for comparison and future machine-learning applications. The identified outliers were retained because they may represent genuine logistics events rather than erroneous observations.

In [ ]:
# FINAL VALIDATION

In [18]:
print("========== FINAL VALIDATION ==========")

print("Final records:", len(delivery_orders))

print(
    "Duplicate rows:",
    delivery_orders.duplicated().sum()
)

print("\nMissing values:")
print(delivery_orders.isna().sum())

print(
    "\nNegative delivery days:",
    (delivery_orders["actual_delivery_days"] < 0).sum()
)

print(
    "\nOutlier records:",
    (delivery_orders["delivery_time_outlier"] == "Outlier").sum()
)

print(
    "\nDelivery performance:"
)
print(delivery_orders["delivery_performance"].value_counts())

print(
    "\nNormalized range:",
    delivery_orders["actual_delivery_days_normalized"].min(),
    "to",
    delivery_orders["actual_delivery_days_normalized"].max()
)

========== FINAL VALIDATION ==========
Final records: 96470
Duplicate rows: 0

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date        1
order_delivered_customer_date       0
order_estimated_delivery_date       0
actual_delivery_days                0
estimated_delivery_days             0
delay_days                          0
delivery_performance                0
delivery_time_outlier               0
actual_delivery_days_normalized     0
dtype: int64

Negative delivery days: 0

Outlier records: 5022

Delivery performance:
delivery_performance
On Time    89163
Delayed     7307
Name: count, dtype: int64

Normalized range: 0.0 to 1.0


In [ ]:
# Saving Clean Dataset 

In [19]:
delivery_orders.to_csv(
    "logistics_delivery_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


##### All critical fields required for delivery-time analysis were complete. Fifteen missing values remained in non-critical operational timestamp fields (order_approved_at and order_delivered_carrier_date), and these records were retained because they did not affect the delivery-time calculations.